In [2]:
import pandas as pd
import numpy as np

In [3]:
dfo = pd.read_csv('dfo_masterlist.csv', encoding='latin1')
emdat = pd.read_csv('historical_dataset.csv', encoding='latin1')

In [8]:
dfo.isnull().sum()

Register__       0
Annual_DFO       0
Glide__       3687
Country__c     326
Other         3730
Nations       3927
Affected      4029
Detailed_L     404
Rivers        2182
Began          326
Ended          326
Duration_i       0
Dead             0
Displaced        0
Damage__US       0
Main_cause     382
Severity__       0
Affected_s       0
Magnitude        0
Centroid_X       0
Centroid_Y       0
"x"_if_act    4018
M_6              0
Total_annu       0
M_4              0
Total_an_1       0
Date_Began     326
Total_floo       0
Total_fl_1       0
F31           4029
F32           4029
F33           4029
F34           4029
F35           4029
F36           4029
dtype: int64

In [9]:
emdat.isnull().sum()

DisNo.                                           0
Historic                                         0
Classification Key                               0
Disaster Group                                   0
Disaster Subgroup                                0
Disaster Type                                    0
Disaster Subtype                                 0
External IDs                                 12522
Event Name                                   11548
ISO                                              0
Country                                          0
Subregion                                        0
Region                                           0
Location                                       798
Origin                                       12585
Associated Types                             13203
OFDA/BHA Response                                0
Appeal                                           0
Declaration                                      0
AID Contribution ('000 US$)    

In [15]:
# cleaning and prepare EM-DAT 

# Filter the dataset to only include Floods
emdat_floods = emdat[emdat['Disaster Type'] == 'Flood'].copy()

# Ensure Year and Month are integers for accurate matching
emdat_floods['Merge_Year'] = emdat_floods['Start Year'].astype(int)
emdat_floods['Merge_Month'] = emdat_floods['Start Month'].fillna(0).astype(int)

# Create a cleaned country column (lowercase, no extra spaces) to ensure perfect matching
emdat_floods['Country_Clean'] = emdat_floods['Country'].astype(str).str.strip().str.lower()

# Select only the columns we need for the ML model
emdat_cols = [
    'Merge_Year', 'Merge_Month', 'Country_Clean', 'Country', 'ISO', 
    'Disaster Subtype', 'Total Deaths', 'No. Affected', 'Total Affected', 
    "Total Damage ('000 US$)"
]
emdat_clean = emdat_floods[emdat_cols].copy()
emdat_clean

,Merge_Year,Merge_Month,Country_Clean,Country,ISO,Disaster Subtype,Total Deaths,No. Affected,Total Affected,Total Damage ('000 US$)
1,2026,3,kenya,Kenya,KEN,Flood (General),59.0,NaN,27.0,NaN
3,2026,4,afghanistan,Afghanistan,AFG,Flood (General),18.0,73300.0,73309.0,NaN
11,2026,5,democratic republic of the congo,Democratic Republic of the Congo,COD,Flood (General),6.0,79000.0,79000.0,NaN
16,2025,8,equatorial guinea,Equatorial Guinea,GNQ,Flood (General),NaN,23115.0,23115.0,NaN
31,2025,3,argentina,Argentina,ARG,Flood (General),125.0,235900.0,236700.0,375000.0
...,...,...,...,...,...,...,...,...,...,...
16900,2019,10,niger,Niger,NER,Flood (General),NaN,23000.0,23000.0,NaN
16901,2019,10,united republic of tanzania,United Republic of Tanzania,TZA,Flood (General),14.0,NaN,NaN,NaN
16902,2019,9,senegal,Senegal,SEN,Flood (General),6.0,8919.0,8968.0,NaN
16911,2019,11,france,France,FRA,Flood (General),5.0,100.0,102.0,NaN


In [ ]:
# clean and prepare DFO 

# Drop rows where the start date is completely missing
dfo_clean = dfo.dropna(subset=['Began']).copy()

# Convert the 'Began' string into a standard pandas datetime format
dfo_clean['Began_Date'] = pd.to_datetime(dfo_clean['Began'], errors='coerce')

# Drop any rows where the date conversion failed
dfo_clean = dfo_clean.dropna(subset=['Began_Date']).copy()

# # Extract the Year and Month to match with EM-DAT
# dfo_clean['Merge_Year'] = dfo_clean['Began_Date'].dt.year.astype(int)
# dfo_clean['Merge_Month'] = dfo_clean['Began_Date'].dt.month.astype(int)

# # Standardize the Country names just like we did for EM-DAT
# dfo_clean['Country_Clean'] = dfo_clean['Country__c'].astype(str).str.strip().str.lower()

# # Select the geospatial and severity columns
# dfo_cols = [
#     'Merge_Year', 'Merge_Month', 'Country_Clean',
#     'Centroid_X', 'Centroid_Y', 'Duration_i', 'Dead', 'Displaced', 
#     'Severity__', 'Affected_s', 'Magnitude'
# ]
# dfo_selected = dfo_clean[dfo_cols].copy()

      Register__  Annual_DFO Glide__     Country__c         Other Nations  \
0         3703.0       114.0     NaN          Niger  Burkina Faso    Chad   
1         3702.0       113.0     NaN  United States           NaN     NaN   
2         3701.0       112.0     NaN    North Korea           NaN     NaN   
3         3700.0       111.0     NaN          India           NaN     NaN   
4         3699.0       110.0     NaN          China           NaN     NaN   
...          ...         ...     ...            ...           ...     ...   
3698         5.0         5.0     NaN     Mozambique           NaN     NaN   
3699         4.0         4.0     NaN      Indonesia           NaN     NaN   
3700         3.0         3.0     NaN    Philippines           NaN     NaN   
3701         2.0         2.0     NaN         Brazil           NaN     NaN   
3702         1.0         1.0     NaN        Algeria           NaN     NaN   

      Affected                                         Detailed_L Rivers  \

,Register__,Annual_DFO,Glide__,Country__c,Other,Nations,Affected,Detailed_L,Rivers,Began,...,Date_Began,Total_floo,Total_fl_1,F31,F32,F33,F34,F35,F36,Began_Date
0,3703.0,114.0,NaN,Niger,Burkina Faso,Chad,NaN,"Niamey, northern Chad",NaN,2010-08-01,...,2010-08-01,896.0,3171.0,NaN,NaN,NaN,NaN,NaN,NaN,2010-08-01
1,3702.0,113.0,NaN,United States,NaN,NaN,NaN,"State of Iowa, Ames, Colfax towns",NaN,2010-08-10,...,2010-08-10,895.0,3170.0,NaN,NaN,NaN,NaN,NaN,NaN,2010-08-10
2,3701.0,112.0,NaN,North Korea,NaN,NaN,NaN,"Eastern province of South Hamkyong, northweste...",NaN,2010-07-27,...,2010-07-27,895.0,3169.0,NaN,NaN,NaN,NaN,NaN,NaN,2010-07-27
3,3700.0,111.0,NaN,India,NaN,NaN,NaN,"Kashmir,",NaN,2010-08-06,...,2010-08-06,895.0,3168.0,NaN,NaN,NaN,NaN,NaN,NaN,2010-08-06
4,3699.0,110.0,NaN,China,NaN,NaN,NaN,Entire communities in Gansu province's Zhouqu ...,NaN,2010-07-27,...,2010-07-27,894.0,3167.0,NaN,NaN,NaN,NaN,NaN,NaN,2010-07-27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3698,5.0,5.0,NaN,Mozambique,NaN,NaN,NaN,"Provinces: Natal, Maputo; Rivers: Nkomati, Oma...",NaN,1985-02-09,...,1985-02-09,1.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,1985-02-09
3699,4.0,4.0,NaN,Indonesia,NaN,NaN,NaN,Region: Northern Sulawesi; Towns: Gorontalo Re...,NaN,1985-02-04,...,1985-02-04,1.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1985-02-04
3700,3.0,3.0,NaN,Philippines,NaN,NaN,NaN,Towns: Tanjay a Pamplona,NaN,1985-01-20,...,1985-01-20,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,1985-01-20
3701,2.0,2.0,NaN,Brazil,NaN,NaN,NaN,"States: Rio de Janeiro, Minas Gerais a Espirit...",NaN,1985-01-15,...,1985-01-15,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1985-01-15
